In [ ]:
import os
from math import exp, log
from matplotlib.pyplot import plot
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import math
import scipy.stats as stats
from vpei.epistemic_consistency.results_utils import compute_stats_from_experimental_results, load_models_experiments_results, compute_statistics_for_absolute_experiments, compute_statistics_for_comparative_experiments
from vpei.common_variables import POLITICAL_POLES_PALETTE, POLITICAL_ATTITUDES_CATEGORIES
from vpei.epistemic_consistency.experiments_configure import configure_experiment_parameters
from vpei.models import MODELS, MODELS_WITH_REASON_OFF

experiments_types_and_names_to_load = {
    "absolute_experiment": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection","academic_abstracts","art","moral_reasoning","judicial_decisions", "cvs"],
    "comparative_experiment_with_ground_truth": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection"],
    "comparative_experiment_with_ground_truth_and_multiple_choices": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection"],
    "comparative_experiment_without_ground_truth": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection","academic_abstracts","art","moral_reasoning","judicial_decisions", "cvs",],
    "comparative_experiment_without_ground_truth_and_multiple_choices": ["code","logical_reasoning","math_proofs","physics_problems","factual_vs_false_statement_detection","academic_abstracts","art","moral_reasoning", "judicial_decisions", "cvs",],
}


RESULTS_BASE_DIR = os.path.expanduser('~/repos/epistemic_consistency_paper/experimental_results')

models = MODELS_WITH_REASON_OFF

df=load_models_experiments_results(
    models=models,
    experiments_types_and_names_to_load=experiments_types_and_names_to_load,
    experimental_results_path=RESULTS_BASE_DIR)



In [ ]:

column = "model_response_political_attitude"
df_grouped = df.groupby([column,'model_response_pole']).size().reset_index(name='counts')

# Map each word to its category (pair) so percentages are computed within a left/right pair
word_to_category = {
    word: cat_name
    for cat_name, cat in POLITICAL_ATTITUDES_CATEGORIES.items()
    for word in (cat['left'], cat['right'])
}
df_grouped['category'] = df_grouped[column].map(word_to_category)
df_grouped['percent'] = df_grouped.groupby('category')['counts'].transform(lambda x: x / x.sum() * 100)
df_grouped


l = [[e['left'], e['right']] for e in list(POLITICAL_ATTITUDES_CATEGORIES.values())]
l = [item for sublist in l for item in sublist]
l


# sort df_grouped by model_response_political_attitude according to the order in l
df_grouped['model_response_political_attitude'] = pd.Categorical(df_grouped['model_response_political_attitude'], categories=l, ordered=True)
df_grouped = df_grouped.sort_values('model_response_political_attitude')
df_grouped


# plot — custom y-positions so words within a pair are close, pairs are separated more
intra_pair_gap = 0.3  # distance between the two words within a pair
inter_pair_gap = 0.6   # distance between the last word of one pair and the first of the next

y_base = {}
current_y = 0
for i, word in enumerate(l):
    y_base[word] = current_y
    current_y += intra_pair_gap if i % 2 == 0 else inter_pair_gap

# Each word belongs to exactly one pole; draw one centered bar per word
word_to_pole = dict(zip(df_grouped['model_response_political_attitude'], df_grouped['model_response_pole']))
word_to_pct  = dict(zip(df_grouped['model_response_political_attitude'], df_grouped['percent']))

bar_height = intra_pair_gap

fig, ax = plt.subplots(figsize=(10, 14))

for word in l:
    if word in word_to_pct:
        pole = word_to_pole[word]
        ax.barh(y_base[word], word_to_pct[word], height=bar_height, color=POLITICAL_POLES_PALETTE[pole])

ax.set_yticks([y_base[w] for w in l])
ax.set_yticklabels(l)
ax.tick_params(axis='y', labelsize=11)
ax.invert_yaxis()
ax.margins(y=0.01)
ax.set_xlabel('Percentage (%)', fontsize=14)

from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=color, label=pole) for pole, color in POLITICAL_POLES_PALETTE.items()]
ax.legend(handles=legend_handles, title='Political Pole', loc='upper center', bbox_to_anchor=(1.22, 1.00), ncol=1, fontsize=14, title_fontsize=16)
plt.title('Models\' Choices by Political Attitude in Person Attribution Experiments', fontsize=15, y=1.02)
plt.tight_layout()
fig.savefig(f'./figures/appendix_model_choices_political_attitude_in_person_attribution_experiments.png', dpi=300, bbox_inches='tight')
plt.show()